In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import add_self_loops, degree

import warnings
warnings.filterwarnings('ignore')

/opt/anaconda3/envs/topicgraph_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def add_self_loops(edge_index, num_nodes):
    self_loops = torch.arange(num_nodes)
    self_loops = torch.stack([self_loops, self_loops], dim=0)  # [[0,1,2],[0,1,2]]
    edge_index = torch.cat([edge_index, self_loops], dim=1)
    return edge_index, None

def degree(index, num_nodes, dtype=None):
    deg = torch.zeros(num_nodes, dtype=dtype)
    ones = torch.ones(index.size(0), dtype=dtype)
    deg.scatter_add_(0, index, ones)
    return deg


class GCNConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.W = nn.Linear(in_channels, out_channels, bias=False)

    def forward(self, x, edge_index, num_nodes):
        print("\n[0] 입력 x")
        print(x)
        print("shape:", x.shape)

        print("\n[1] 입력 edge_index")
        print(edge_index)
        print("shape:", edge_index.shape)

        edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
        print("\n[2] self-loop 추가 후 edge_index")
        print(edge_index)
        print("shape:", edge_index.shape)

        row, col = edge_index
        print("\n[3] row (source)")
        print(row)
        print("shape:", row.shape)

        print("\n[4] col (target)")
        print(col)
        print("shape:", col.shape)

        deg = degree(col, num_nodes=num_nodes, dtype=x.dtype)
        print("\n[5] degree(col 기준)")
        print(deg)
        print("shape:", deg.shape)

        deg_inv_sqrt = deg.pow(-0.5)
        print("\n[6] deg_inv_sqrt")
        print(deg_inv_sqrt)
        print("shape:", deg_inv_sqrt.shape)

        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        print("\n[7] norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]")
        print(norm)
        print("shape:", norm.shape)

        x = self.W(x)
        print("\n[8] 선형변환 후 x = W(x)")
        print(x)
        print("shape:", x.shape)

        xr = x[row]
        print("\n[9] x[row]  # 각 edge의 source 노드 feature")
        print(xr)
        print("shape:", xr.shape)

        norm_u = norm.unsqueeze(-1)
        print("\n[10] norm.unsqueeze(-1)")
        print(norm_u)
        print("shape:", norm_u.shape)

        src = norm_u * xr
        print("\n[11] src = norm.unsqueeze(-1) * x[row]")
        print(src)
        print("shape:", src.shape)

        out = torch.zeros(num_nodes, x.size(1), device=x.device)
        print("\n[12] 초기 out")
        print(out)
        print("shape:", out.shape)

        index = col.unsqueeze(-1).expand_as(xr)
        print("\n[13] index = col.unsqueeze(-1).expand_as(x[row])")
        print(index)
        print("shape:", index.shape)

        print("\n[14] edge별로 out[target] += src")
        for e in range(edge_index.size(1)):
            print(f"edge {e}: {row[e].item()} -> {col[e].item()}, "
                  f"src={src[e].tolist()}")

        out.scatter_add_(dim=0, index=index, src=src)

        print("\n[15] 최종 out")
        print(out)
        print("shape:", out.shape)

        return out


# -----------------------------
# 예시 데이터
# -----------------------------
# 노드 3개, feature 1차원
x = torch.tensor([
    [1.0],   # node 0
    [2.0],   # node 1
    [3.0]    # node 2
])

# 무방향 그래프 0-1, 1-2
# 실제 edge_index는 양방향으로 넣음
edge_index = torch.tensor([
    [0, 1, 1, 2],
    [1, 0, 2, 1]
])

num_nodes = 3

conv = GCNConv(in_channels=1, out_channels=1)

# 가중치를 직접 고정: W = [[2.0]]
with torch.no_grad():
    conv.W.weight[:] = torch.tensor([[2.0]])

out = conv(x, edge_index, num_nodes)


[0] 입력 x
tensor([[1.],
        [2.],
        [3.]])
shape: torch.Size([3, 1])

[1] 입력 edge_index
tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])
shape: torch.Size([2, 4])

[2] self-loop 추가 후 edge_index
tensor([[0, 1, 1, 2, 0, 1, 2],
        [1, 0, 2, 1, 0, 1, 2]])
shape: torch.Size([2, 7])

[3] row (source)
tensor([0, 1, 1, 2, 0, 1, 2])
shape: torch.Size([7])

[4] col (target)
tensor([1, 0, 2, 1, 0, 1, 2])
shape: torch.Size([7])

[5] degree(col 기준)
tensor([2., 3., 2.])
shape: torch.Size([3])

[6] deg_inv_sqrt
tensor([0.7071, 0.5774, 0.7071])
shape: torch.Size([3])

[7] norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
tensor([0.4082, 0.4082, 0.4082, 0.4082, 0.5000, 0.3333, 0.5000])
shape: torch.Size([7])

[8] 선형변환 후 x = W(x)
tensor([[2.],
        [4.],
        [6.]], grad_fn=<MmBackward0>)
shape: torch.Size([3, 1])

[9] x[row]  # 각 edge의 source 노드 feature
tensor([[2.],
        [4.],
        [4.],
        [6.],
        [2.],
        [4.],
        [6.]], grad_fn=<IndexBackward0>)
shape: torch.S